# $\nu_e$ CC Inclusive — Systematics

Loads pre-saved `sel_topo` and the mcnu weights file.  
**Outputs:** `saved_syst/` (NPZ) · `plots_syst/` (PNG)

In [1]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

import os, sys, gc, warnings
import numpy as np
import pandas as pd
# import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

CAFPYANA_WD = '/home/castalyf/cafpyana'
for p in [CAFPYANA_WD, CAFPYANA_WD + '/pyanalib']:
    if p not in sys.path:
        sys.path.insert(0, p)

from analysis_village.unfolding.covariance import (
    get_covariance_matrix_self, get_covariance_matrix)
from analysis_village.unfolding.utils import plot_univ_hists, plot_heatmap

# ── Paths ──────────────────────────────────────────────────────────────────────
DF_OUT_DIR   = './nuecc_dfs'
SYST_DIR     = './saved_syst'
PLOT_DIR     = './plots_syst'
TARGET_POT   = 6.6e20
TITLE        = r'SBND $\nu_e$ CC Inclusive'
POT_LABEL    = r'$6.6\times10^{20}$ POT'

# sel_qual: the ONLY file needed for all analysis (POT, events, truth)
SEL_FILE = f'{DF_OUT_DIR}/selected_nuecc_qual.df'

# DF_FILE: ONLY used in Cell 3 (mct_index for weight alignment).
# MUST be the same file used when sel_qual was generated.
# DF_FILE      = '/home/castalyf/cafpyana_out/mc1e20_nueCC_test_small.df'
DF_FILE      = '/home/castalyf/cafpyana_out/mc1e20_nueCC_test_small.df'


# WEIGHTS_FILE: universe weights (flux/GENIE reweighting)
WEIGHTS_FILE = '/home/castalyf/cafpyana_out/mc1e20_nueCC_sys_test_small.df'

for d in [SYST_DIR, PLOT_DIR,
          f'{PLOT_DIR}/univ', f'{PLOT_DIR}/cov', f'{PLOT_DIR}/fracunc']:
    os.makedirs(d, exist_ok=True)
print('Paths OK')


Paths OK


In [2]:
def _load_hdf_key(hdf_file, key_prefix):
    with pd.HDFStore(hdf_file, mode='r') as store:
        keys = store.keys()
        if '/split' in keys:
            n = int(store['split']['n_split'].iloc[0])
            return pd.concat([store[f'{key_prefix}_{k}'] for k in range(n)])
        matching = sorted(k for k in keys if k.lstrip('/').startswith(key_prefix.lstrip('/')))
        if not matching:
            return pd.DataFrame()
        return pd.concat([store[k] for k in matching])

histpotdf = _load_hdf_key(DF_FILE, 'histpotdf')
# statsdf   = _load_hdf_key(DF_FILE, 'stats')
statsdf   = _load_hdf_key(DF_FILE, 'truth_info')

# ── Load sel_qual (the only file we need for syst analysis) ───────────────────
sel_topo = pd.read_hdf(SEL_FILE)

# Deduplicate safety guard
if sel_topo.index.duplicated().any():
    print(f'WARNING: {sel_topo.index.duplicated().sum()} duplicate rows — deduplicating')
    sel_topo = sel_topo[~sel_topo.index.duplicated(keep='first')]

print(f'sel_topo shape : {sel_topo.shape}')
print(f'sel_topo cols  : {list(sel_topo.columns)}')

# ── pot_scale: read from sel_qual — avoids any DF_FILE mismatch ───────────────
pot_scale = float(sel_topo['pot_scale'].iloc[0])

# ── n_true_signal: all is_sig==True rows (presel denominator) ─────────────────
n_true_signal = int(sel_topo['is_sig'].sum())

# ── TARGET_POT back-computed so labels are consistent ─────────────────────────
# (sel_qual stores pot_scale = TARGET_POT / sample_POT, so we use stored value)
print(f'\npot_scale     : {pot_scale:.4f}')
print(f'n_true_signal : {n_true_signal:,}  (signal in presel — efficiency denominator)')

# Quick sanity check: signal at each stage
for col in [c for c in sel_topo.columns if c.startswith('sel_')]:
    n_sig = int((sel_topo[col] & sel_topo['is_sig']).sum())
    n_tot = int(sel_topo[col].sum())
    print(f'  {col:<35} sig={n_sig:>6,}  total={n_tot:>9,}  '
          f'weighted_sig={n_sig*pot_scale:.1f}')


sel_topo shape : (44070, 16)
sel_topo cols  : ['truth_cat', 'is_sig', 'reco_ke', 'reco_costheta', 'reco_p', 'true_ke', 'true_costheta', 'true_p', 'sel_precut', 'sel_valid_flashmatch', 'sel_fiducial', 'sel_single_electron', 'sel_electron_primary_score', 'sel_electron_pid_score', 'sel_vertex_distance', 'pot_scale']

pot_scale     : 87.6736
n_true_signal : 174  (signal in presel — efficiency denominator)
  sel_precut                          sig=   174  total=   44,070  weighted_sig=15255.2
  sel_valid_flashmatch                sig=   174  total=   44,070  weighted_sig=15255.2
  sel_fiducial                        sig=   174  total=   44,070  weighted_sig=15255.2
  sel_single_electron                 sig=   167  total=    1,215  weighted_sig=14641.5
  sel_electron_primary_score          sig=   132  total=      661  weighted_sig=11572.9
  sel_electron_pid_score              sig=   113  total=      202  weighted_sig=9907.1
  sel_vertex_distance                 sig=   108  total=      174  w

## 2.Load Weights & Universe Columns

In [3]:
mcnu_df = _load_hdf_key(WEIGHTS_FILE, 'mcnu')
print(f'mcnu_df shape : {mcnu_df.shape}')
print(f'mcnu_df index : {mcnu_df.index.names}')

def _is_univ(c):
    return isinstance(c, tuple) and any('univ' in str(p).lower() for p in c)
def _matches(c, kws):
    return isinstance(c, tuple) and any(
        any(kw in str(p).lower() for kw in kws) for p in c)

all_cols   = list(mcnu_df.columns)
bnb_cols   = [c for c in all_cols if _is_univ(c) and _matches(c,
              ['flux','bnb','beam','expskin','horncurrent','kplus','kminus',
               'kzero','piplus','piminus','nucleon','pion'])]
genie_cols = [c for c in all_cols if _is_univ(c) and _matches(c, ['genie','xsr','knob'])]
if not bnb_cols:
    bnb_cols   = [c for c in all_cols if _is_univ(c) and _matches(c, ['flux'])]
if not genie_cols:
    genie_cols = [c for c in all_cols if _is_univ(c) and _matches(c, ['genie'])]

print(f'BNB cols  : {len(bnb_cols)}')
print(f'GENIE cols: {len(genie_cols)}')

# Extract weight columns immediately, then free the full mcnu_df


AttributeError: Attribute 'axis0_variety' does not exist in node: '/split'

## 3. Align Weights to sel_topo

Reads **only** the `mct_index` column from `evtdf` (chunked). Frees everything temporary afterwards.

In [ ]:

print('Reading mct_index column from evtdf ...')
with pd.HDFStore(DF_FILE, mode='r') as store:
    keys = store.keys()
    if '/split' in keys:
        n = int(store['split']['n_split'].iloc[0])
        evt_keys = [f'/evt_{k}' for k in range(n)]
    else:
        evt_keys = sorted(k for k in keys if k.lstrip('/').startswith('evt'))

with pd.HDFStore(DF_FILE, mode='r') as store:
    sample_cols = store[evt_keys[0]].columns.tolist()

mct_col = next((c for c in sample_cols
                if 'mct_index' in str(c) and 'dlp_true' in str(c)), None)
if mct_col is None:
    mct_col = next((c for c in sample_cols if 'mct_index' in str(c)), None)
print(f'mct_index column: {mct_col}')

mct_frames = []
for ek in tqdm(evt_keys, desc='chunks'):
    with pd.HDFStore(DF_FILE, mode='r') as store:
        chunk = store[ek][[mct_col]]
    il = chunk.index.names[:-1] if chunk.index.nlevels > 1 else chunk.index.names
    mct_frames.append(chunk.groupby(level=il).first()); del chunk
mct_inter = pd.concat(mct_frames); del mct_frames; gc.collect()
print(f'mct_inter shape: {mct_inter.shape}')

# Merge with mcnu weights
wgt_cols  = bnb_cols + genie_cols
mcnu_sub  = mcnu_df[wgt_cols].copy() if wgt_cols else pd.DataFrame(index=mcnu_df.index)
del mcnu_df; gc.collect()

mcnu_idx   = mcnu_sub.index.names
inter_idx  = list(mct_inter.index.names)
mct_r  = mct_inter.reset_index()
mcnu_r = mcnu_sub.reset_index()

# ── Flatten MultiIndex columns to plain strings before merging ─────────────────
def _flat(cols):
    return ['|'.join(str(p) for p in c).strip('|') if isinstance(c, tuple) else str(c)
            for c in cols]

mct_r.columns  = _flat(mct_r.columns)
mcnu_r.columns = _flat(mcnu_r.columns)
print(f'mcnu_sub shape: {mcnu_sub.shape}')

# Re-derive key names after flattening
inter_idx_flat = _flat(inter_idx)      # ['__ntuple', 'entry', 'rec.dlp..index']
mcnu_idx_flat  = _flat(mcnu_idx)       # ['__ntuple', 'entry', 'rec.mc.nu..index']
mct_col_flat   = _flat([mct_col])[0]   # 'rec|dlp_true|mct_index||'
wgt_cols_flat  = _flat(wgt_cols)

merged = mct_r.merge(
    mcnu_r,
    left_on  = [inter_idx_flat[0], inter_idx_flat[1], mct_col_flat],
    right_on = [mcnu_idx_flat[0],  mcnu_idx_flat[1],  mcnu_idx_flat[2]],
    how='left',
)
merged = merged.set_index(inter_idx_flat)

# Restore original tuple column names so downstream cells are unaffected
wgt_aligned = merged[wgt_cols_flat].copy()
wgt_aligned.columns    = wgt_cols   # put tuple cols back
wgt_aligned.index.names = inter_idx  # restore original index names

del mct_r, mcnu_r, merged, mct_inter, mcnu_sub; gc.collect()

bnb_cols_aligned   = [c for c in wgt_aligned.columns if c in set(bnb_cols)]
genie_cols_aligned = [c for c in wgt_aligned.columns if c in set(genie_cols)]
print(f'wgt_aligned: {wgt_aligned.shape}  BNB={len(bnb_cols_aligned)}  GENIE={len(genie_cols_aligned)}')


## 4. Binning & Variable Configs

In [ ]:
RECO_BINS   = np.linspace(0, 2000, 8)
TRUE_BINS   = np.linspace(0, 2000, 8)
COS_BINS    = np.linspace(-1, 1, 11)
FINAL_STAGE = 'sel_vertex_distance'   # or 'sel_single_electron' for qual cuts

class VarConfig:
    def __init__(self, name, bins, xlabel):
        self.name          = name
        self.bins          = bins
        self.bin_centers   = 0.5 * (bins[:-1] + bins[1:])
        self.var_plot_name = xlabel
        self.var_labels    = [xlabel, xlabel, xlabel.replace('Reco','True')]
        self.pot_label     = POT_LABEL

VAR_CFGS = [
    VarConfig('reco_ke',       RECO_BINS, r'Reco leading-$e^-$ KE [MeV]'),
    VarConfig('reco_costheta', COS_BINS,  r'Reco leading-$e^-$ $\cos\theta$'),
]
print('VarConfigs ready:')
for v in VAR_CFGS:
    print(f'  {v.name}  n_bins={len(v.bins)-1}')


## 5. CV Histograms & Response Matrix

In [ ]:
_eps = 1e-8

def build_cv(sel_df, stage_col, var_col, bins, ps,
             true_col=None, true_bins=None):
    sel  = sel_df[sel_df[stage_col]]
    smk  = (sel['truth_cat'] == 0).values
    bmk  = sel['truth_cat'].between(1, 6).values
    reco = sel[var_col].clip(bins[0], bins[-1] - _eps).values
    w    = np.full(len(sel), ps)
    sig_cv, _ = np.histogram(reco[smk], bins=bins, weights=w[smk])
    bkg_cv, _ = np.histogram(reco[bmk], bins=bins, weights=w[bmk])
    out = dict(sig_cv=sig_cv, bkg_cv=bkg_cv)
    if true_col and true_bins is not None:
        tv = sel.loc[smk, true_col].dropna()
        true_cv, _ = np.histogram(
            tv.clip(true_bins[0], true_bins[-1] - _eps),
            bins=true_bins, weights=np.full(len(tv), ps))
        valid = smk & sel[var_col].notna().values & sel[true_col].notna().values
        r2d, _, _ = np.histogram2d(
            sel.loc[valid, true_col].clip(true_bins[0], true_bins[-1] - _eps),
            sel.loc[valid, var_col].clip(bins[0], bins[-1] - _eps),
            bins=[true_bins, bins])
        out['true_sig_cv'] = true_cv
        out['response']    = r2d / r2d.sum(axis=1, keepdims=True).clip(1)
    return out

cv_results = {}
for vcfg in VAR_CFGS:
    tc = 'true_ke' if vcfg.name == 'reco_ke' else 'true_costheta'
    tb = TRUE_BINS if vcfg.name == 'reco_ke'  else vcfg.bins
    cv_results[vcfg.name] = build_cv(
        sel_topo, FINAL_STAGE, vcfg.name, vcfg.bins, pot_scale, tc, tb)
    cv = cv_results[vcfg.name]
    print(f'{vcfg.name}: sig={cv["sig_cv"].sum():.1f}  bkg={cv["bkg_cv"].sum():.1f}')
print('CV done.')


## 6. Universe Histograms (Flux & GENIE)

In [ ]:

def build_univ_hists(sel_df, stage_col, wgt_aligned, univ_cols,
                     var_col, bins, ps, clip=10.0):
    sel  = sel_df[sel_df[stage_col]]
    smk  = (sel['truth_cat'] == 0).values
    bmk  = sel['truth_cat'].between(1, 6).values
    reco = sel[var_col].clip(bins[0], bins[-1] - 1e-8).values
    nwl  = wgt_aligned.index.nlevels
    sidx = (sel.index.droplevel(list(range(nwl, sel.index.nlevels)))
            if sel.index.nlevels > nwl else sel.index)
    wmat = wgt_aligned[univ_cols].reindex(sidx).values.astype(np.float32)
    nu   = len(univ_cols); nb = len(bins) - 1
    su   = np.zeros((nu, nb)); bu = np.zeros((nu, nb))
    for ui in tqdm(range(nu), desc=var_col, leave=False):
        w = np.where(np.isfinite(wmat[:, ui]), wmat[:, ui].astype(float), 1.0)
        w = np.clip(w, 0.0, clip) * ps
        su[ui], _ = np.histogram(reco[smk], bins=bins, weights=w[smk])
        bu[ui], _ = np.histogram(reco[bmk], bins=bins, weights=w[bmk])
    return su, bu

SRC_LABEL    = {'flux': 'BNB Flux', 'genie': 'GENIE', 'mcstat': 'MCstat'}
univ_results = {}

for vcfg in VAR_CFGS:
    univ_results[vcfg.name] = {}
    if bnb_cols_aligned:
        print(f'[{vcfg.name}] BNB ...')
        su, bu = build_univ_hists(sel_topo, FINAL_STAGE, wgt_aligned,
                                  bnb_cols_aligned, vcfg.name, vcfg.bins, pot_scale)
        univ_results[vcfg.name]['flux'] = {'sig': su, 'bkg': bu}
    if genie_cols_aligned:
        print(f'[{vcfg.name}] GENIE ...')
        su, bu = build_univ_hists(sel_topo, FINAL_STAGE, wgt_aligned,
                                  genie_cols_aligned, vcfg.name, vcfg.bins, pot_scale)
        univ_results[vcfg.name]['genie'] = {'sig': su, 'bkg': bu}
    gc.collect()
print('Flux/GENIE universes done.')


## 7. MCstat Universes (Poisson resampling)

In [ ]:

from numpy.random import Generator, PCG64, SeedSequence

def build_mcstat_univs(sel_df, stage_col, var_col, bins, ps, nu=100, seed=42):
    sel  = sel_df[sel_df[stage_col]]
    smk  = (sel['truth_cat'] == 0).values
    bmk  = sel['truth_cat'].between(1, 6).values
    reco = sel[var_col].clip(bins[0], bins[-1] - 1e-8).values
    ne   = len(sel); ss = SeedSequence(seed); kids = ss.spawn(nu)
    nb   = len(bins) - 1; su = np.zeros((nu, nb)); bu = np.zeros((nu, nb))
    for u in tqdm(range(nu), desc=f'{var_col} MCstat', leave=False):
        w = Generator(PCG64(kids[u])).poisson(1.0, size=ne) * ps
        su[u], _ = np.histogram(reco[smk], bins=bins, weights=w[smk])
        bu[u], _ = np.histogram(reco[bmk], bins=bins, weights=w[bmk])
    return su, bu

for vcfg in VAR_CFGS:
    print(f'[{vcfg.name}] MCstat ...')
    su, bu = build_mcstat_univs(sel_topo, FINAL_STAGE, vcfg.name, vcfg.bins, pot_scale)
    univ_results[vcfg.name]['mcstat'] = {'sig': su, 'bkg': bu}

del wgt_aligned, bnb_cols_aligned, genie_cols_aligned; gc.collect()
print('MCstat done. Weights freed.')


## 8. Covariance Matrices

In [ ]:

def _mat(block):
    return block.get('cov', block) if isinstance(block, dict) else block

def compute_covs(su, bu, s_cv, b_cv):
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', RuntimeWarning)
        return {
            'cov_ms_ms': get_covariance_matrix_self(su, s_cv),
            'cov_bs_bs': get_covariance_matrix_self(bu, b_cv),
            'cov_ms_bs': get_covariance_matrix(su, s_cv, bu, b_cv),
            'cov_bs_ms': get_covariance_matrix(bu, b_cv, su, s_cv),
        }

cov_results = {}
for vcfg in VAR_CFGS:
    cov_results[vcfg.name] = {}
    cv = cv_results[vcfg.name]
    for src, ud in univ_results[vcfg.name].items():
        print(f'[{vcfg.name}] cov: {src} ...')
        cov_results[vcfg.name][src] = compute_covs(
            ud['sig'], ud['bkg'], cv['sig_cv'], cv['bkg_cv'])
print('All covariances computed.')


## 9. Save NPZ Files

In [ ]:

def save_npz(path, cv, covs, extra=None):
    p = dict(ms=cv['sig_cv'], bs=cv['bkg_cv'])
    if 'true_sig_cv' in cv: p['true_signal'] = cv['true_sig_cv']
    if 'response'    in cv: p['response']    = cv['response']
    for bk, bv in covs.items():
        if isinstance(bv, dict):
            for k, v in bv.items():
                if isinstance(v, np.ndarray): p[f'{bk}_{k}'] = v
        else:
            p[bk] = bv
    if extra: p.update(extra)
    np.savez(path, **p)
    print(f'  {os.path.basename(path)}  ({os.path.getsize(path)/1024:.0f} KB)')

for vcfg in VAR_CFGS:
    vd = os.path.join(SYST_DIR, vcfg.name); os.makedirs(vd, exist_ok=True)
    cv = cv_results[vcfg.name]
    for src, covs in cov_results[vcfg.name].items():
        save_npz(os.path.join(vd, f'{src}_cov_matrices.npz'), cv, covs)
    if len(cov_results[vcfg.name]) > 1:
        ts = sum(_mat(cov_results[vcfg.name][s]['cov_ms_ms'])
                 for s in cov_results[vcfg.name])
        tb = sum(_mat(cov_results[vcfg.name][s]['cov_bs_bs'])
                 for s in cov_results[vcfg.name])
        save_npz(os.path.join(vd, 'total_cov_matrices.npz'), cv, {},
                 extra=dict(total_cov_ms_ms=ts, total_cov_bs_bs=tb))
print(f'\nAll NPZ saved to {SYST_DIR}/')


## 10. Plots: Universe Spreads

In [ ]:
def savefig(fig, name, sub=''):
    d = os.path.join(PLOT_DIR, sub) if sub else PLOT_DIR
    os.makedirs(d, exist_ok=True)
    fig.savefig(os.path.join(d, f'{name}.png'), dpi=150, bbox_inches='tight')
    display(fig)       # show inline in notebook
    plt.close(fig)

for vcfg in VAR_CFGS:
    cv = cv_results[vcfg.name]
    for src, ud in univ_results[vcfg.name].items():
        for categ, cv_arr, ua in [
            ('Signal',     cv['sig_cv'], ud['sig']),
            ('Background', cv['bkg_cv'], ud['bkg']),
        ]:
            try:
                plot_univ_hists(ua, cv_arr, SRC_LABEL[src], vcfg, categ_name=categ)
                fig = plt.gcf()   # grab whatever was just created
                savefig(fig, f'{vcfg.name}_{src}_{categ.lower()}_univs', 'univ')
            except Exception as e:
                print(f'  [WARN] {vcfg.name}/{src}/{categ}: {e}')
                plt.close('all')

print('Universe plots -> plots_syst/univ/')

## 11. Plots: Covariance Heatmaps

In [ ]:
import re

def format_heatmap_value(v):
    if np.isnan(v): return ""
    av = abs(v)
    if av == 0: return "0"
    if av < 1:
        return f"{v:.4f}"
    elif av < 1e3:
        return f"{v:.2f}"
    mantissa, exponent = f"{v:.2e}".split("e")
    return rf"${mantissa}\times10^{{{int(exponent)}}}$"

def plot_heatmap_custom(matrix, title, vcfg, axis_labels=None,
                        cmap='viridis', vmin=None, vmax=None):
    """Heatmap with bin-range tick labels and formatted cell annotations."""
    try:
        from analysis_village.unfolding.utils import bin_range_labels, get_text_color
        raw     = bin_range_labels(vcfg.bins)
        t_lbls  = [re.sub(r'(\d+)\.0+(?!\d)', r'\1', l) for l in raw]
        _get_tc = get_text_color
    except Exception:
        t_lbls  = [f'{vcfg.bins[i]:.0f}–{vcfg.bins[i+1]:.0f}'
                   for i in range(len(vcfg.bins)-1)]
        _get_tc = lambda v: 'white'

    n        = len(vcfg.bins) - 1
    unif     = np.arange(n + 1, dtype=float)
    tick_pos = 0.5 * (unif[:-1] + unif[1:])
    extent   = [unif[0], unif[-1], unif[0], unif[-1]]

    fig, ax = plt.subplots(figsize=(max(6, n*0.8), max(5, n*0.7)))
    im = ax.imshow(matrix, extent=extent, origin='lower',
                   cmap=cmap, aspect='auto', vmin=vmin, vmax=vmax)
    cbar = plt.colorbar(im, ax=ax)
    cbar.ax.tick_params(labelsize=9)

    ax.set_xticks(tick_pos)
    ax.set_xticklabels(t_lbls, rotation=45, ha='right', fontsize=8)
    ax.set_yticks(tick_pos)
    ax.set_yticklabels(t_lbls, fontsize=8)
    ax.tick_params(axis='both', labelsize=8)

    xlbl, ylbl = axis_labels if axis_labels else [vcfg.var_plot_name]*2
    ax.set_xlabel(xlbl, fontsize=11)
    ax.set_ylabel(ylbl, fontsize=11)
    ax.set_title(title, fontsize=12)

    # Cell annotations
    fsize = max(5, 9 - n//3)   # shrink font for more bins
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            v = matrix[i, j]
            if not np.isnan(v):
                try:    tc = _get_tc(v)
                except: tc = 'white'
                ax.text(tick_pos[j], tick_pos[i],
                        format_heatmap_value(v),
                        ha='center', va='center',
                        color=tc, fontsize=fsize)
    fig.tight_layout()
    return fig, ax

In [ ]:
BLOCK_LABELS = {
    'cov_ms_ms': ('Cov(sig,sig)', 'viridis',  None,  None),
    'cov_bs_bs': ('Cov(bkg,bkg)', 'viridis',  None,  None),
    'cov_ms_bs': ('Cov(sig,bkg)', 'RdBu_r',   None,  None),
    'cov_bs_ms': ('Cov(bkg,sig)', 'RdBu_r',   None,  None),
}

for vcfg in VAR_CFGS:
    for src, covs in cov_results[vcfg.name].items():
        for bk, (bl, cmap, vmin, vmax) in BLOCK_LABELS.items():
            bv       = covs[bk]
            cov_mat  = _mat(bv)
            corr_mat = bv.get('corr', None) if isinstance(bv, dict) else None

            try:
                fig, _ = plot_heatmap_custom(
                    cov_mat, f'{SRC_LABEL[src]} — {bl} [Cov]',
                    vcfg, cmap=cmap, vmin=vmin, vmax=vmax)
                display(fig)
                savefig(fig, f'{vcfg.name}_{src}_{bk}_cov', 'cov')
            except Exception as e:
                print(f'  [WARN] cov {vcfg.name}/{src}/{bk}: {e}')

            if corr_mat is not None:
                try:
                    fig, _ = plot_heatmap_custom(
                        corr_mat, f'{SRC_LABEL[src]} — {bl} [Corr]',
                        vcfg, cmap='coolwarm', vmin=-1, vmax=1)
                    display(fig)
                    savefig(fig, f'{vcfg.name}_{src}_{bk}_corr', 'cov')
                except Exception as e:
                    print(f'  [WARN] corr {vcfg.name}/{src}/{bk}: {e}')

print('Heatmaps -> plots_syst/cov/')

## 12. Plots: Fractional Uncertainties per Bin

In [ ]:

COLORS = {'flux': '#2196F3', 'genie': '#F44336', 'mcstat': '#4CAF50'}

def plot_fracunc(vcfg, cv_arr, cov_results_var, block_key, categ):
    fig, ax = plt.subplots(figsize=(8, 4))
    total_var = np.zeros(len(vcfg.bin_centers))
    for src, covs in cov_results_var.items():
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            diag = np.diag(_mat(covs[block_key])).clip(0)
            frac = np.where(cv_arr > 0, np.sqrt(diag) / cv_arr, 0.0)
        xe = np.append(vcfg.bins[:-1], vcfg.bins[-1])
        ye = np.append(frac, frac[-1])
        ax.step(xe, ye, where='post', label=SRC_LABEL[src],
                color=COLORS.get(src, 'gray'), lw=1.8)
        total_var += diag
    tot = np.where(cv_arr > 0, np.sqrt(total_var) / cv_arr, 0.0)
    ax.step(np.append(vcfg.bins[:-1], vcfg.bins[-1]),
            np.append(tot, tot[-1]),
            where='post', label='Total', color='black', lw=2.2, ls='--')
    ax.set_xlabel(vcfg.var_plot_name, fontsize=12)
    ax.set_ylabel('Fractional uncertainty', fontsize=12)
    ax.set_title(f'{TITLE} - {categ}', fontsize=12)
    ax.legend(fontsize=10, ncol=2)
    ax.set_xlim(vcfg.bins[0], vcfg.bins[-1]); ax.set_ylim(bottom=0)
    ax.grid(axis='y', alpha=0.3)
    ax.text(0.99, 0.97, vcfg.pot_label, transform=ax.transAxes,
            ha='right', va='top', fontsize=10, color='gray')
    fig.tight_layout(); return fig

for vcfg in VAR_CFGS:
    cv = cv_results[vcfg.name]; covs_var = cov_results[vcfg.name]
    for bk, categ, cv_arr in [
        ('cov_ms_ms', 'Signal',     cv['sig_cv']),
        ('cov_bs_bs', 'Background', cv['bkg_cv']),
    ]:
        fig = plot_fracunc(vcfg, cv_arr, covs_var, bk, categ)
        savefig(fig, f'{vcfg.name}_fracunc_{categ.lower()}', 'fracunc')
print('Fracunc plots -> plots_syst/fracunc/')


## 13. CV Stacked Histogram + Systematic Band

In [ ]:
print(f"sel_topo shape        : {sel_topo.shape}")
print(f"Index names           : {sel_topo.index.names}")
print(f"Index nlevels         : {sel_topo.index.nlevels}")
stage_sum = sel_topo[FINAL_STAGE].sum()
sig_sum   = (sel_topo[FINAL_STAGE] & (sel_topo['truth_cat'] == 0)
             & sel_topo['reco_ke'].notna()).sum()
print(f"Passing {FINAL_STAGE}: {stage_sum}")
print(f"Signal passing stage  : {sig_sum}")
print(f"Expected weighted sig : {sig_sum * pot_scale:.1f}  (should be ~9322)")

In [ ]:
import nue_helpers as nh

DISPLAY_CATS  = [0, 1, 2, 3, 4, 5, 6]
FALLBACK_CLRS = ['#1f77b4','#aec7e8','#ff7f0e','#ffbb78','#2ca02c','#98df8a','#d62728']
FALLBACK_LBLS = [r'$\nu_e$ CC FV', r'$\nu_e$ CC out FV',
                 r'$\nu_\mu$ CC+$\pi^0$', r'NC $\pi^0$',
                 r'Other $\nu_\mu$ CC', 'Other NC', 'Cosmic']
_labels = [nh.CAT_LABELS[c] for c in DISPLAY_CATS] 
_colors = [nh.CAT_COLORS[c] for c in DISPLAY_CATS] 

# Use more bins for display (separate from covariance bins)
DISP_KE_BINS  = np.linspace(0, 2000, 13)   # 40 bins, 50 MeV wide
DISP_COS_BINS = np.linspace(-1, 1,   13)   # 40 bins

DISP_BINS = {'reco_ke': DISP_KE_BINS, 'reco_costheta': DISP_COS_BINS}

for vcfg in VAR_CFGS:
    cv       = cv_results[vcfg.name]
    covs_var = cov_results[vcfg.name]
    total_var = sum(np.diag(_mat(covs_var[s]['cov_ms_ms'])).clip(0) for s in covs_var)

    # Fractional unc from covariance bins → interpolate onto display bins
    cov_centers  = vcfg.bin_centers
    disp_bins    = DISP_BINS[vcfg.name]
    disp_centers = 0.5 * (disp_bins[:-1] + disp_bins[1:])
    frac_cv      = np.where(cv['sig_cv'] > 0,
                            np.sqrt(total_var.clip(0)) / cv['sig_cv'], 0.0)
    frac_disp    = np.interp(disp_centers, cov_centers, frac_cv)

    # ── Stacked histogram (nh handles weights & NaN correctly) ────────────
    def _cat(col, cat):
        m = (sel_topo[FINAL_STAGE]
             & (sel_topo['truth_cat'] == cat)
             & sel_topo['reco_ke'].notna()
             & sel_topo['reco_costheta'].notna())
        return sel_topo.loc[m, col].values   # NO clip — numpy excludes OOB naturally

    fig, ax = plt.subplots(figsize=(8, 6))
    nh.plot_stacked_hist(
        series_list=[_cat(vcfg.name, c) for c in DISPLAY_CATS],
        labels=_labels, colors=_colors,
        bins=disp_bins,
        weights=pot_scale,
        xlabel=vcfg.var_plot_name,
        title=TITLE,
        pot_label=POT_LABEL,
        ax=ax,
        invert_stack_order=True,
        show_counts=True,
        show_percentage=True,
    )

    # ── Syst uncertainty band (on total MC = sig+bkg) ─────────────────────
    # Recompute total MC per display bin for band baseline
    total_disp = np.zeros(len(disp_centers))
    for cat in DISPLAY_CATS:
        m = (sel_topo[FINAL_STAGE] & (sel_topo['truth_cat'] == cat)
             & sel_topo['reco_ke'].notna() & sel_topo['reco_costheta'].notna())
        v, _ = np.histogram(sel_topo.loc[m, vcfg.name].values,
                            bins=disp_bins,
                            weights=np.full(m.sum(), pot_scale))
        total_disp += v

    unc_disp = frac_disp * total_disp
    bw = np.diff(disp_bins)
    ax.bar(disp_bins[:-1], 2 * unc_disp, bottom=total_disp - unc_disp,
           width=bw, align='edge', alpha=0.3, color='gray',
           hatch='///', label='Syst. unc.', linewidth=0)

    handles, lbls = ax.get_legend_handles_labels()
    ax.legend(handles, lbls, fontsize=7, ncol=2, loc='upper right',
              frameon=True, framealpha=0.85, edgecolor='none')
    fig.tight_layout()
    savefig(fig, f'{vcfg.name}_cv_with_syst_band')

print('Stacked+syst -> plots_syst/')

In [ ]:
from nue_xsec_helpers import make_nuecc_binning2d, plot_nuecc_differential_stacked

binning2d_nue = make_nuecc_binning2d()
print(f'n_costheta_bins  : {binning2d_nue.n_costheta_bins}')
print(f'costheta_bins    : {binning2d_nue.diff_costheta_bins}')
print(f'KE bins (slice 0): {binning2d_nue.diff_momentum_bins_2d[0]} MeV')

cv_ke        = cv_results['reco_ke']
covs_ke      = cov_results['reco_ke']
total_var_ke = sum(np.diag(_mat(covs_ke[s]['cov_ms_ms'])).clip(0) for s in covs_ke)
frac_unc_ke  = np.where(cv_ke['sig_cv'] > 0,
                        np.sqrt(total_var_ke) / cv_ke['sig_cv'], 0.0)

fig, axs = plot_nuecc_differential_stacked(
    sel_df               = sel_topo,
    stage_col            = FINAL_STAGE,
    ke_col               = 'reco_ke',
    costheta_col         = 'reco_costheta',
    truth_cat_col        = 'truth_cat',
    binning2d            = binning2d_nue,
    display_cats         = DISPLAY_CATS,
    labels               = _labels,
    colors               = _colors,
    pot_scale            = pot_scale,
    frac_unc_ke          = frac_unc_ke,
    reco_ke_bins         = VAR_CFGS[0].bins,
    title                = fr'{TITLE} — $\cos\theta_e$ slices',
    pot_label            = POT_LABEL,
    plot_dir             = PLOT_DIR,
    filename             = 'nuecc_differential_costheta_slices',
    plot_stacked_hist_fn = nh.plot_stacked_hist,
)

In [ ]:

print(f'\n{"="*68}')
print(f'{"Variable":<16} {"Source":<10} {"Category":<14} {"Mean frac. unc.":>16}')
print(f'{"="*68}')
for vcfg in VAR_CFGS:
    cv = cv_results[vcfg.name]
    for src, covs in cov_results[vcfg.name].items():
        for bk, categ, cv_arr in [
            ('cov_ms_ms', 'Signal',     cv['sig_cv']),
            ('cov_bs_bs', 'Background', cv['bkg_cv']),
        ]:
            with warnings.catch_warnings():
                warnings.simplefilter('ignore')
                frac = np.where(cv_arr > 0,
                    np.sqrt(np.diag(_mat(covs[bk])).clip(0)) / cv_arr, np.nan)
            fu = float(np.nanmean(frac))
            print(f'  {vcfg.name:<14} {src:<10} {categ:<14} {fu*100:>14.2f}%')
print(f'{"="*68}')
print(f'\nNPZ files -> {SYST_DIR}/')
for vcfg in VAR_CFGS:
    vd = f'{SYST_DIR}/{vcfg.name}'
    if os.path.isdir(vd):
        print(f'  {vcfg.name}: {os.listdir(vd)}')
print(f'Plots     -> {PLOT_DIR}/')
for sub in ['univ', 'cov', 'fracunc']:
    sd = f'{PLOT_DIR}/{sub}'
    if os.path.isdir(sd):
        print(f'  {sub}/: {len(os.listdir(sd))} files')


In [ ]:
# # Check what fraction of weights are NaN in wgt_aligned
# nan_frac = wgt_aligned[bnb_cols_aligned].isna().mean().mean()
# print(f'NaN fraction in BNB weights: {nan_frac:.1%}')

# # Check a few weight values for matched rows
# matched = wgt_aligned[bnb_cols_aligned[0]].notna()
# print(f'Matched interactions: {matched.sum()} / {len(matched)}')
# print(wgt_aligned.loc[matched, bnb_cols_aligned[:3]].describe())

In [ ]:
for vcfg in VAR_CFGS:
    cv = cv_results[vcfg.name]
    scale = pot_scale  # your pot_scale from the selection notebook
    print(f"\n{vcfg.name} — estimated raw MC events per bin:")
    print(f"  signal : {np.round(cv['sig_cv'] / scale, 1)}")
    print(f"  bkg    : {np.round(cv['bkg_cv'] / scale, 1)}")